In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import h5py
import sys
# sys.path.insert(0, '/Users/smericks/Desktop/StrongLensing/padma_timedelays/fasttdc/')
sys.path.insert(0, '/Users/padmavenkatraman/Documents/StrongLensing/fastTDC/fasttdc/')
import Modeling.MassModels.paltas_preds as paltas_preds

Step #0: produce the approximate image models from NPE and their associated paltas-formatted metadata_truth.csv

In [4]:
# step 1: convert time-delay catalog to paltas format
multiband_df = pd.read_csv('DataVectors/deblended_time_delays.csv')
for key in multiband_df.keys():
    print(key)

Unnamed: 0
dataset
subfolder
bh_mass
closest_image_separation
deflector_redshift
eddington_ratio
inclination_angle
kappa_0
kappa_1
kappa_star_0
kappa_star_1
num_images
observer_frame_sf
observer_frame_tau
shear_0
shear_1
source_redshift
time_delays_0
time_delays_1
true_magnitudes_0
true_magnitudes_1
true_magnitudes_2
true_magnitudes_3
true_magnitudes_4
true_magnitudes_5
ID
deflector_mass_theta_E
deflector_mass_gamma
deflector_mass_center_x
deflector_mass_center_y
deflector_mass_e1
deflector_mass_e2
deflector_mass_gamma1
deflector_mass_gamma2
deflector_mass_ra_0
deflector_mass_dec_0
deflector_mass_kappa
ps_u_mag_true
deflector_light_u_magnitude
deflector_light_u_R_sersic
deflector_light_u_n_sersic
deflector_light_u_e1
deflector_light_u_e2
deflector_light_u_center_x
deflector_light_u_center_y
point_source_light_u_ra_image_0
point_source_light_u_ra_image_1
point_source_light_u_dec_image_0
point_source_light_u_dec_image_1
point_source_light_u_magnitude_0
point_source_light_u_magnitude_1
ex

In [5]:
# WE'RE USING i-BAND FOR EVERYTHING!!

key_translator = {
    # LENS MASS
    'main_deflector_parameters_theta_E': 'deflector_mass_theta_E',
    'main_deflector_parameters_gamma1': 'deflector_mass_gamma1',
    'main_deflector_parameters_gamma2': 'deflector_mass_gamma2',
    'main_deflector_parameters_gamma': 'deflector_mass_gamma',
    'main_deflector_parameters_e1': 'deflector_mass_e1',
    'main_deflector_parameters_e2': 'deflector_mass_e2',
    'main_deflector_parameters_center_x': 'deflector_mass_center_x',
    'main_deflector_parameters_center_y': 'deflector_mass_center_y',
    'main_deflector_parameters_z_lens': 'deflector_redshift',


    # LENS LIGHT
    'lens_light_parameters_R_sersic': 'deflector_light_i_R_sersic',
    'lens_light_parameters_center_x': 'deflector_light_i_center_x',
    'lens_light_parameters_center_y': 'deflector_light_i_center_y',
    'lens_light_parameters_e1': 'deflector_light_i_e1',
    'lens_light_parameters_e2': 'deflector_light_i_e2',
    'lens_light_parameters_mag_app': 'deflector_light_i_magnitude',
    'lens_light_parameters_n_sersic': 'deflector_light_i_n_sersic',
    'lens_light_parameters_z_source': 'deflector_redshift',


    # SOURCE
    'source_parameters_R_sersic': 'extended_source_light_i_R_sersic',
    'source_parameters_center_x': 'extended_source_light_i_center_x',
    'source_parameters_center_y': 'extended_source_light_i_center_y',
    'source_parameters_e1': 'extended_source_light_i_e1',
    'source_parameters_e2': 'extended_source_light_i_e2',
    'source_parameters_mag_app': 'extended_source_light_i_magnitude',
    'source_parameters_n_sersic': 'extended_source_light_i_n_sersic',
    'source_parameters_z_source': 'source_redshift',

    # POINT SOURCE
    'point_source_parameters_mag_app': 'ps_i_mag_true',
    'point_source_parameters_x_point_source': 'extended_source_light_i_center_x',
    'point_source_parameters_y_point_source': 'extended_source_light_i_center_y',
    'point_source_parameters_z_point_source': 'source_redshift',
}

paltas_df = pd.DataFrame()
paltas_df['catalog_idx'] = multiband_df['dataset']
for key in key_translator.keys():
    paltas_df[key] = multiband_df[key_translator[key]]

In [6]:
paltas_df

,catalog_idx,main_deflector_parameters_theta_E,main_deflector_parameters_gamma1,main_deflector_parameters_gamma2,main_deflector_parameters_gamma,main_deflector_parameters_e1,main_deflector_parameters_e2,main_deflector_parameters_center_x,main_deflector_parameters_center_y,main_deflector_parameters_z_lens,...,source_parameters_center_y,source_parameters_e1,source_parameters_e2,source_parameters_mag_app,source_parameters_n_sersic,source_parameters_z_source,point_source_parameters_mag_app,point_source_parameters_x_point_source,point_source_parameters_y_point_source,point_source_parameters_z_point_source
0,deblcdouble653,0.842323,-0.023288,-0.000753,2.113863,-0.237521,-0.038308,0.012326,-0.029470,0.646613,...,0.057181,0.014264,0.291800,25.649106,1.0,3.272536,24.912352,0.210889,0.057181,3.272536
1,deblcquad134,0.971630,0.035391,-0.009625,1.736185,-0.170785,-0.326370,-0.009240,0.055357,0.614830,...,0.164555,-0.065196,-0.192626,25.331095,1.0,2.450944,23.238164,0.179163,0.164555,2.450944
2,deblcdouble91,0.277571,-0.004609,0.021804,2.017652,-0.053444,-0.048492,-0.035690,-0.028720,1.063872,...,-0.146141,0.038693,0.146282,25.165559,1.0,2.115709,22.939993,-0.066862,-0.146141,2.115709
3,deblcdouble98,0.781619,-0.004262,0.006744,1.870207,0.040738,0.106863,-0.037636,0.012262,0.352567,...,0.291249,-0.002199,0.105372,25.230671,1.0,1.308415,24.713128,-0.016219,0.291249,1.308415
4,deblcdouble23,0.283372,0.070019,0.033466,2.122834,0.004416,0.056147,0.030015,-0.037275,0.917997,...,0.220767,0.296584,-0.298256,24.156366,1.0,2.995327,21.149275,0.094125,0.220767,2.995327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,deblcdouble469,0.241687,0.045797,0.022826,1.751160,-0.059482,0.034708,0.015969,-0.013192,0.663422,...,-0.065058,-0.015236,0.175534,24.965437,1.0,3.591399,23.324596,-0.043842,-0.065058,3.591399
150,deblcquad421,0.621044,-0.001539,0.023887,1.783938,0.121605,0.047324,0.005917,-0.002083,0.234840,...,-0.013145,-0.302826,0.030140,25.069167,1.0,2.178241,23.993843,-0.036128,-0.013145,2.178241
151,deblcquad285,0.434632,0.010201,0.002586,1.894661,0.186710,-0.252781,-0.037414,0.042149,0.388462,...,0.055331,-0.015046,-0.005963,27.477721,1.0,2.642741,21.110757,0.003012,0.055331,2.642741
152,deblctriple73,0.350717,-0.006663,0.011140,1.499827,-0.310230,-0.019298,0.027453,0.011626,0.528276,...,-0.073133,-0.183880,-0.104177,22.962137,1.0,1.349862,21.601274,0.017175,-0.073133,1.349862


In [8]:
# # step 2: construct HST-quality paltas network predictor
# hst_preds = paltas_preds.PaltasPreds('Modeling/MassModels/hst_training_config.py',
#     numpix=165,
#     model_weights='Modeling/MassModels/xresnet34_hst_epoch72.h5',
#     model_norms='Modeling/MassModels/hst_norms.csv')

In [ ]:
# step 3 produce predicted image models
images_hst, metadata_list_hst, y_pred_hst, std_pred_hst, cov_pred_hst = hst_preds.preds_from_params(paltas_df)

In [ ]:
multiband_df.shape

In [ ]:
def save_h5(h5_path,catalog_idxs,images,mu_npe,cov_npe):
    h5f = h5py.File(h5_path, 'w')
    h5f.create_dataset('catalog_idx', data=catalog_idxs)
    h5f.create_dataset('images_array', data=images)
    h5f.create_dataset('mu_npe',data=mu_npe)
    h5f.create_dataset('cov_npe',data=cov_npe)
    h5f.close()

# save full paltas formatted metadata with catalog_idx
metadata_df_hst = pd.DataFrame(metadata_list_hst)
metadata_df_hst['catalog_idx'] = paltas_df['catalog_idx']


# GOLD
metadata_df_hst.to_csv('DataVectors/truth_metadata.csv', index=False)
save_h5('DataVectors/hst_image_models.h5',
        catalog_idxs=metadata_df_hst.loc[:,'catalog_idx'].to_numpy(),
        images=images_hst,
        mu_npe=y_pred_hst,
        cov_npe=cov_pred_hst)